# 0 — Prerequisites: image processing + QuPath segmentation

This pipeline starts from **QuPath segmentation outputs**, which are themselves
downstream of raw-image processing. The two upstream stages are *not* part of this
repo:

1. **Image processing** (illumination correction, stitching, deconvolution, EDF,
   registration, autofluorescence removal) — use the author's
   [KINTSUGI](https://github.com/smith6jt-cop/KINTSUGI) pipeline (STAR Protocols 2025).
2. **Cell segmentation** in [QuPath](https://qupath.github.io/) (InstanSeg / StarDist),
   then export **two artifacts** this pipeline consumes:
   - the per-cell **measurement CSV** (`Cellmeasurements.csv`), and
   - full-resolution per-cell **GeoJSON** boundaries (feature `id` == detection UUID).

Set these paths in the **Parameters** cell below (this notebook is self-contained — no `config.ini`).

In [ ]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)
# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few


cfg = PipelineConfig(
    data_dir       = REPO / "data",
    images_dir     = pathlib.Path("/home/smith6jt/IO60panc2nd/Images"),
    cells_csv      = pathlib.Path("/home/smith6jt/IO60panc2nd/Cellmeasurements.csv"),
    donor_metadata = pathlib.Path("/home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx"),
)
print("data_dir      :", cfg.data_dir)
print("images_dir    :", cfg.images_dir)
print("cells_csv     :", cfg.cells_csv)
print("donor_metadata:", cfg.donor_metadata)
print("geojson_dir   :", cfg.geojson_dir)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))


## Export per-cell GeoJSON (REDSEA input)

Run the QuPath Groovy exporter **once per image** (GUI closed). It writes
`data/redsea_scratch/geojson/cells__<image>.geojson`, which `redsea` rasterizes.

In [ ]:
# Shown for reference — run these in a shell with QuPath installed, not in the kernel.
print(r'''
# single image (headless):
QuPath script --project <project.qpproj> --image "<image name>" \\
    scripts/groovy/export_cells_geojson.groovy

# or batch every image missing a GeoJSON:
bash scripts/senior_export_new_geojsons.sh   # (adapt paths for your project)
''')
print('geojson dir:', cfg.geojson_dir)

In [ ]:
# Verify the GeoJSON export (REDSEA input). Not required once data/cells_redsea/ already exists.
geo = cfg.geojson_dir
n_geo = len(list(geo.glob("cells__*.geojson"))) if geo.exists() else 0
print(f"GeoJSON exports: {n_geo}" + (f"   ({geo})" if n_geo else "   -- MISSING"))
if n_geo == 0:
    print("NOTE: REDSEA (step 2) needs these. If data/cells_redsea/ already exists, REDSEA is skipped\n"
          "      and the GeoJSONs are not required for a load-and-validate run.")